# Graph Sampling Pipeline

## Import Libraries

In [1]:
from pathlib import Path
from collections import deque
import pandas as pd
import random
import duckdb
import os
import time
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import pyarrow.parquet as pq
import pyarrow as pa
import glob
import json

## User Config

In [2]:
LABEL_PATH = Path("./raw_datasets/label.csv")
SPLIT_PATH = Path("./raw_datasets/split.csv")
EDGE_CSV_PATH = Path("./raw_datasets/edge.csv")

EDGE_PARQUET_PATH = Path("./datasets/edge.parquet")
USER_PARQUET_PATH = Path("./datasets/user.parquet")

USER_JSON_PATH = Path("./raw_datasets/user.json")

OUTPUT_DIR = Path("./datasets/final_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_USERS_FINAL = OUTPUT_DIR / "df_users_final.parquet"
OUTPUT_EDGES_FINAL = OUTPUT_DIR / "df_edges_final.parquet"

RANDOM_STATE = 42
TOTAL_N = 40_000
HUMAN_RATIO = 0.61
HUMAN_SEED_N = 14_000
BOT_SEED_N = 10_000

USE_SPLIT_FILTER = None

SOCIAL_RELATIONS = {"following", "followers", "followed"}
USER_RELEVANT_RELATIONS = {
	"followers",
	"following",
	"post",
	"pinned",
	"like",
	"own",
	"membership",
	"followed",
}
FINAL_GRAPH_RELATIONS = {
	"contain",
	"discuss",
	"followed",
	"followers",
	"following",
	"like",
	"membership",
	"mentioned",
	"own",
	"pinned",
	"post",
	"quoted",
	"replied_to",
	"retweeted",
}

## Convert CSV to parquet

In [3]:
con = duckdb.connect()
con.execute("INSTALL json; LOAD json;")
con.execute(f"PRAGMA threads={os.cpu_count()}")
con.execute("SET memory_limit='10GB'")
con.execute(f"PRAGMA temp_directory='{Path.cwd() / '.duckdb_tmp'}'")
con.execute("PRAGMA enable_progress_bar")

def csv_to_parquet(csv_path: str, parquet_path: str) -> Path:
	csv_path = Path(csv_path)
	parquet_path = Path(parquet_path)
	parquet_path.parent.mkdir(parents=True, exist_ok=True)

	if parquet_path.exists():
		print(f"Already exists, skipping: {parquet_path}")
		return parquet_path

	t0 = time.perf_counter()
	con.execute(f"""
		COPY (
			SELECT * FROM read_csv('{csv_path.as_posix()}',
				sample_size=-1,
				parallel=true)
		)
		TO '{parquet_path.as_posix()}'
		(FORMAT PARQUET, COMPRESSION 'zstd', COMPRESSION_LEVEL 3, ROW_GROUP_SIZE 1000000)
	""")
	dt = time.perf_counter() - t0

	in_mb = csv_path.stat().st_size / (1024 * 1024)
	out_mb = parquet_path.stat().st_size / (1024 * 1024)
	print(f"{csv_path.name}: {in_mb:,.1f} MB -> {out_mb:,.1f} MB in {dt:,.1f}s ({in_mb/dt:,.1f} MB/s)")

	return parquet_path

## Load Label.csv and Split.csv

In [4]:
def load_label_split(label_path: Path, split_path: Path) -> pd.DataFrame:
	df_label = pd.read_csv(label_path)
	df_split = pd.read_csv(split_path)
	
	if not {"id", "label"}.issubset(df_label.columns):
		raise ValueError("label.csv must contain columns: id, label")
	if not {"id", "split"}.issubset(df_split.columns):
		raise ValueError("split.csv must contain columns: id, split")

	for name, df in [("label", df_label), ("split", df_split)]:
		if df["id"].isna().any():
			raise ValueError(f"{name}.csv has null id values")

	df_label["label"] = df_label["label"].str.strip().str.lower()
	df_split["split"] = df_split["split"].str.strip().str.lower()

	merged = df_label.merge(df_split, on="id", how="left", validate="one_to_one")
	missing = merged["split"].isna().sum()
	if missing:
		raise ValueError(f"{missing} ids in label.csv have no split assignment")
	merged["label"] = merged["label"].astype("category")
	merged["split"] = merged["split"].astype("category")

	return merged

## Helper Functions

### Convert user.json to parquet and loading

In [5]:
def user_json_to_parquet(json_path: Path, parquet_path: Path) -> Path:
	json_path, parquet_path = Path(json_path), Path(parquet_path)
	parquet_path.parent.mkdir(parents=True, exist_ok=True)

	if parquet_path.exists():
		print(f"Already exists, skipping: {parquet_path}")
		return parquet_path

	t0 = time.perf_counter()
	con.execute(f"""
		COPY (
			SELECT * FROM read_json_auto(
				'{json_path.as_posix()}',
				sample_size=-1,
				maximum_object_size=104857600
			)
		)
		TO '{parquet_path.as_posix()}'
		(FORMAT PARQUET, COMPRESSION 'zstd', ROW_GROUP_SIZE 1000000)
	""")
	print(f"user.json -> parquet in {time.perf_counter() - t0:,.1f}s")
	return parquet_path

In [6]:
def load_user_parquet_filtered(parquet_path: Path, sampled_ids) -> pd.DataFrame:
	con.register("sampled", pd.DataFrame({"id": pd.Series(sorted(sampled_ids), dtype="string")}))
	df = con.execute(f"""
		SELECT u.*
		FROM read_parquet('{Path(parquet_path).as_posix()}') u
		WHERE u.id IN (SELECT id FROM sampled)
	""").df()
	con.unregister("sampled")

	s = df["id"].astype("string")
	df["id"] = s.where(s.str.startswith("u"), "u" + s)
	return df

In [7]:
def flatten_user_metadata(df_user_metadata: pd.DataFrame) -> pd.DataFrame:
	records = df_user_metadata.to_dict(orient="records")
	return pd.json_normalize(records)

### Graph Views for sampling

In [8]:
def _sql_in(values: set[str]) -> str:
	return ", ".join("'" + v.replace("'", "''") + "'" for v in sorted(values))


def compute_participation_degree(
	edge_parquet_path: Path,
	user_relevant_relations: set[str],
) -> pd.DataFrame:
	t0 = time.perf_counter()
	df_degree = con.execute(f"""
		WITH e AS (
			SELECT source_id, target_id
			FROM read_parquet('{Path(edge_parquet_path).as_posix()}')
			WHERE relation IN ({_sql_in(user_relevant_relations)})
			  AND source_id <> target_id
		),
		participants AS (
			SELECT source_id AS id FROM e WHERE starts_with(source_id, 'u')
			UNION ALL
			SELECT target_id AS id FROM e WHERE starts_with(target_id, 'u')
		)
		SELECT id, COUNT(*)::INTEGER AS participation_degree
		FROM participants
		GROUP BY id
	""").df()
	print(f"participation degree: {len(df_degree):,} users in {time.perf_counter()-t0:,.1f}s")
	return df_degree


def build_social_adjacency(
	edge_parquet_path: Path,
	social_relations: set[str],
	eligible_ids: pd.Series,
) -> dict[str, list[str]]:
	con.register("eligible", pd.DataFrame({"id": eligible_ids.astype(str).unique()}))

	t0 = time.perf_counter()
	df_adj = con.execute(f"""
		WITH s AS (
			SELECT source_id, target_id
			FROM read_parquet('{Path(edge_parquet_path).as_posix()}')
			WHERE relation IN ({_sql_in(social_relations)})
			  AND source_id <> target_id
			  AND starts_with(source_id, 'u')
			  AND starts_with(target_id, 'u')
			  AND source_id IN (SELECT id FROM eligible)
			  AND target_id IN (SELECT id FROM eligible)
		),
		undirected AS (
			SELECT source_id AS id, target_id AS nbr FROM s
			UNION
			SELECT target_id AS id, source_id AS nbr FROM s
		)
		SELECT id, list(nbr ORDER BY nbr) AS neighbours
		FROM undirected
		GROUP BY id
	""").df()
	con.unregister("eligible")

	social_adj = {row.id: list(row.neighbours) for row in df_adj.itertuples(index=False)}
	print(f"social adjacency: {len(social_adj):,} nodes in {time.perf_counter()-t0:,.1f}s")
	return social_adj

def build_sampling_base(
	df_base: pd.DataFrame,
	df_degree: pd.DataFrame,
	split_filter: str | None = None
) -> pd.DataFrame:
	df = df_base.merge(df_degree, on="id", how="left")
	df["participation_degree"] = df["participation_degree"].fillna(0).astype(int)

	if split_filter is not None:
		df = df[df["split"] == split_filter].copy()

	df = df[df["participation_degree"] > 0].copy()
	return df

### Sampling using seed based expansion

In [ ]:
def sample_balanced_seeds(
	df_sampling: pd.DataFrame,
	human_seed_n: int,
	bot_seed_n: int,
	random_state: int = 42
) -> pd.DataFrame:
	df_human = df_sampling[df_sampling["label"] == "human"].sort_values(
		"participation_degree", ascending=False
	)
	df_bot = df_sampling[df_sampling["label"] == "bot"].sort_values(
		"participation_degree", ascending=False
	)

	human_pool = df_human.head(min(len(df_human), human_seed_n * 5))
	bot_pool = df_bot.head(min(len(df_bot), bot_seed_n * 5))

	sampled_human = human_pool.sample(n=human_seed_n, random_state=random_state)
	sampled_bot = bot_pool.sample(n=bot_seed_n, random_state=random_state)

	return pd.concat([sampled_human, sampled_bot], axis=0).sample(
		frac=1, random_state=random_state
	).reset_index(drop=True)


def seed_expand_sample(
	df_sampling: pd.DataFrame,
	social_adj: dict,
	total_n: int = 20_000,
	human_seed_n: int = 7_000,
	bot_seed_n: int = 5_000,
	human_target_n: int | None = None,
	bot_target_n: int | None = None,
	random_state: int = 42
) -> pd.DataFrame:
	if human_target_n is None and bot_target_n is None:
		human_target_n = round(total_n * HUMAN_RATIO)
		bot_target_n = total_n - human_target_n
	elif human_target_n is None or bot_target_n is None:
		raise ValueError("Pass both human_target_n and bot_target_n, or neither")

	if human_target_n + bot_target_n != total_n:
		raise ValueError(
			f"human_target_n + bot_target_n ({human_target_n + bot_target_n}) "
			f"must equal total_n ({total_n})"
		)
	if human_seed_n > human_target_n or bot_seed_n > bot_target_n:
		raise ValueError("seed_n cannot exceed target_n for the same label")
	
	rng = random.Random(random_state)

	df_seed = sample_balanced_seeds(
		df_sampling=df_sampling,
		human_seed_n=human_seed_n,
		bot_seed_n=bot_seed_n,
		random_state=random_state
	)

	sampled_ids = set(df_seed["id"])
	frontier = deque(df_seed["id"].tolist())
	id_to_label = df_sampling.set_index("id")["label"].to_dict()

	target_human = human_target_n
	target_bot = bot_target_n

	current_human = int((df_seed["label"] == "human").sum())
	current_bot = int((df_seed["label"] == "bot").sum())

	def can_add(label: str) -> bool:
		if label == "human":
			return current_human < target_human
		if label == "bot":
			return current_bot < target_bot
		return True

	while frontier and len(sampled_ids) < total_n:
		node = frontier.popleft()
		neighbors = list(social_adj.get(node, []))
		rng.shuffle(neighbors)

		for nbr in neighbors:
			if len(sampled_ids) >= total_n:
				break
			if nbr in sampled_ids:
				continue
			nbr_label = id_to_label.get(nbr)
			if nbr_label is None:
				continue
			if not can_add(nbr_label):
				continue

			sampled_ids.add(nbr)
			frontier.append(nbr)

			if nbr_label == "human":
				current_human += 1
			else:
				current_bot += 1

	if len(sampled_ids) < total_n:
		remaining = df_sampling[~df_sampling["id"].isin(sampled_ids)].sort_values(
			"participation_degree", ascending=False
		)

		for row in remaining.itertuples(index=False):
			if len(sampled_ids) >= total_n:
				break
			if row.label == "human" and current_human >= target_human:
				continue
			if row.label == "bot" and current_bot >= target_bot:
				continue

			sampled_ids.add(row.id)
			if row.label == "human":
				current_human += 1
			else:
				current_bot += 1

	if len(sampled_ids) < total_n:
		remaining = df_sampling[~df_sampling["id"].isin(sampled_ids)]
		extra = remaining.sample(
			n=min(total_n - len(sampled_ids), len(remaining)),
			random_state=random_state
		)
		sampled_ids.update(extra["id"].tolist())

	df_sampled = df_sampling[df_sampling["id"].isin(sampled_ids)].copy()

	if len(df_sampled) > total_n:
		human_part = df_sampled[df_sampled["label"] == "human"]
		bot_part = df_sampled[df_sampled["label"] == "bot"]

		n_human = min(len(human_part), human_target_n)
		n_bot = min(len(bot_part), bot_target_n)

		human_part = human_part.sample(n=n_human, random_state=random_state)
		bot_part = bot_part.sample(n=n_bot, random_state=random_state)

		df_sampled = pd.concat([human_part, bot_part], axis=0)

		if len(df_sampled) < total_n:
			leftover = df_sampling[
				df_sampling["id"].isin(sampled_ids) & ~df_sampling["id"].isin(df_sampled["id"])
			]
			extra = leftover.sample(
				n=min(total_n - len(df_sampled), len(leftover)),
				random_state=random_state
			)
			df_sampled = pd.concat([df_sampled, extra], axis=0)

	return df_sampled.reset_index(drop=True)

### Building nodes and edges

In [ ]:
# def load_edges(edge_parquet_path, relations: set[str] | None = None) -> pd.DataFrame:
# 	filters = [("relation", "in", sorted(relations))] if relations is not None else None

# 	table = pq.read_table(
# 		Path(edge_parquet_path).as_posix(),
# 		columns=["source_id", "target_id", "relation"],
# 		filters=filters,
# 	)

# 	table = table.cast(pa.schema([
# 		pa.field("source_id", pa.large_string()),
# 		pa.field("target_id", pa.large_string()),
# 		pa.field("relation",  pa.dictionary(pa.int8(), pa.string())),
# 	]))

# 	df_edges = table.to_pandas(
# 		types_mapper=pd.ArrowDtype,
# 		split_blocks=True,
# 		self_destruct=True,
# 	)
# 	del table

# 	df_edges["relation"] = df_edges["relation"].astype("category")
# 	return df_edges

def build_graph_nodes_and_edges(
	edge_parquet_path: Path,
	sampled_ids,
	final_graph_relations: set[str],
):
	path = Path(edge_parquet_path).as_posix()
	rel_list = _sql_in(final_graph_relations)

	con.register("sampled", pd.DataFrame({"id": pd.Series(sorted(sampled_ids), dtype="string")}))

	t0 = time.perf_counter()

	con.execute(f"""
		CREATE OR REPLACE TEMP TABLE edges AS
		SELECT source_id, target_id, relation
		FROM read_parquet('{path}')
		WHERE relation IN ({rel_list})
	""")

	con.execute("""
		CREATE OR REPLACE TEMP TABLE hop1 AS
		WITH touching AS (
			SELECT source_id AS a, target_id AS b FROM edges
			WHERE source_id IN (SELECT id FROM sampled)
			   OR target_id IN (SELECT id FROM sampled)
		)
		SELECT n FROM (
			SELECT a AS n FROM touching
			UNION
			SELECT b AS n FROM touching
		)
		WHERE NOT starts_with(n, 'u')
	""")

	con.execute("""
		CREATE OR REPLACE TEMP TABLE retained_non_users AS
		WITH touching AS (
			SELECT source_id AS a, target_id AS b FROM edges
			WHERE source_id IN (SELECT n FROM hop1)
			   OR target_id IN (SELECT n FROM hop1)
		),
		hop2 AS (
			SELECT n FROM (
				SELECT a AS n FROM touching
				UNION
				SELECT b AS n FROM touching
			)
			WHERE NOT starts_with(n, 'u')
		)
		SELECT n FROM hop1 UNION SELECT n FROM hop2
	""")

	con.execute("""
		CREATE OR REPLACE TEMP TABLE retained_all AS
		SELECT id AS n FROM sampled
		UNION
		SELECT n FROM retained_non_users
	""")

	df_edges_final = con.execute("""
		SELECT DISTINCT source_id, target_id, relation
		FROM edges
		WHERE source_id IN (SELECT n FROM retained_all)
		  AND target_id IN (SELECT n FROM retained_all)
	""").df()

	stats = con.execute("""
		SELECT
			(SELECT count(*) FROM hop1)                                          AS hop1_nodes,
			(SELECT count(*) FROM retained_non_users)                            AS after_expansion,
			(SELECT count(*) FROM retained_all)                                  AS total_nodes,
			(SELECT count(*) FROM retained_non_users WHERE starts_with(n, 't'))  AS tweets,
			(SELECT count(*) FROM retained_non_users WHERE starts_with(n, 'l'))  AS lists,
			(SELECT count(*) FROM retained_non_users WHERE starts_with(n, 'h'))  AS hashtags
	""").df().iloc[0]

	con.unregister("sampled")
	for t in ["edges", "hop1", "retained_non_users", "retained_all"]:
		con.execute(f"DROP TABLE IF EXISTS {t}")

	print(f"  first hop non-user nodes: {stats.hop1_nodes:,}")
	print(f"  after expansion:          {stats.after_expansion:,} "
		  f"({stats.tweets:,} tweets, {stats.lists:,} lists, {stats.hashtags:,} hashtags)")
	print(f"  total nodes:              {stats.total_nodes:,}")
	print(f"  edges:                    {len(df_edges_final):,} in {time.perf_counter()-t0:,.1f}s")

	return df_edges_final

In [11]:
def validate_relations(edge_parquet_path: Path, *relation_sets: set[str]) -> pd.DataFrame:
	counts = con.execute(f"""
		SELECT relation, COUNT(*) AS n
		FROM read_parquet('{Path(edge_parquet_path).as_posix()}')
		GROUP BY relation
		ORDER BY n DESC
	""").df()

	present = set(counts["relation"])
	expected = set().union(*relation_sets)
	unknown = expected - present

	if unknown:
		raise ValueError(
			f"Relation names not present in edge file: {sorted(unknown)}\n"
			f"Actual relations: {sorted(present)}"
		)

	print(f"All {len(expected)} configured relation names found in edge file.")
	return counts

### Build final user metadata file

In [12]:
def build_users_final_df(
	df_sampled: pd.DataFrame,
	df_user_metadata: pd.DataFrame
) -> pd.DataFrame:
	dup_meta = df_user_metadata["id"].duplicated().sum()
	if dup_meta > 0:
		raise ValueError(
			f"df_user_metadata has {dup_meta} duplicate 'id' values — "
			"merge would create duplicate rows in df_sampled."
		)

	overlap = (set(df_sampled.columns) & set(df_user_metadata.columns)) - {"id"}
	if overlap:
		print(f"Warning: overlapping columns besides 'id', keeping df_sampled's version: {overlap}")
		df_user_metadata = df_user_metadata.drop(columns=list(overlap))

	df_result = df_sampled.merge(df_user_metadata, on="id", how="left")

	missing_ids = set(df_sampled["id"]) - set(df_user_metadata["id"])
	if missing_ids:
		print(f"Warning: {len(missing_ids)} sampled users have no metadata match "
			  f"(e.g. {list(missing_ids)[:5]})")

	return df_result

## Main function

In [ ]:
def main():
	csv_to_parquet(EDGE_CSV_PATH, EDGE_PARQUET_PATH)
	display(con.sql(f"SELECT * FROM read_parquet('{EDGE_PARQUET_PATH.as_posix()}') LIMIT 5"))
	display(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{EDGE_PARQUET_PATH.as_posix()}')"))

	print("Validating relation names...")
	counts = validate_relations(
		EDGE_PARQUET_PATH,
		SOCIAL_RELATIONS,
		USER_RELEVANT_RELATIONS,
		FINAL_GRAPH_RELATIONS,
	)
	display(counts)

	print("Loading label + split...")
	df_base = load_label_split(LABEL_PATH, SPLIT_PATH)

	print("Computing participation degree...")
	df_degree = compute_participation_degree(EDGE_PARQUET_PATH, USER_RELEVANT_RELATIONS)

	print("Building sampling base...")
	df_sampling = build_sampling_base(df_base, df_degree, split_filter=USE_SPLIT_FILTER)
	del df_degree
	print(f"Eligible users: {len(df_sampling):,}")
	print(df_sampling["label"].value_counts())

	print("Building social adjacency...")
	social_adj = build_social_adjacency(
		EDGE_PARQUET_PATH,
		SOCIAL_RELATIONS,
		df_sampling["id"],
	)

	print("Sampling users...")
	df_sampled = seed_expand_sample(
		df_sampling=df_sampling,
		social_adj=social_adj,
		total_n=TOTAL_N,
		human_target_n=24_400,
		bot_target_n=15_600,
		human_seed_n=HUMAN_SEED_N,
		bot_seed_n=BOT_SEED_N,
		random_state=RANDOM_STATE,
	)
	sampled_ids = set(df_sampled["id"])
	del social_adj, df_sampling

	print(f"Sampled {len(df_sampled):,} users")
	print(df_sampled["label"].value_counts())
	print(df_sampled["split"].value_counts())

	print("Converting user.json to parquet (if needed)...")
	user_json_to_parquet(USER_JSON_PATH, USER_PARQUET_PATH)

	print("Loading user metadata (filtered to sampled users)...")
	df_user_metadata = load_user_parquet_filtered(USER_PARQUET_PATH, sampled_ids)
	print(f"  metadata rows: {len(df_user_metadata):,} of {len(sampled_ids):,} sampled")

	print("Flattening nested user metadata fields...")
	df_user_metadata = flatten_user_metadata(df_user_metadata)

	print("Building df_users_final...")
	df_users_final = build_users_final_df(df_sampled, df_user_metadata)
	del df_user_metadata

	print("Building graph nodes and edges...")
	df_edges_final = build_graph_nodes_and_edges(
		EDGE_PARQUET_PATH, sampled_ids, FINAL_GRAPH_RELATIONS
	)

	print("Saving outputs...")
	df_users_final.to_parquet(OUTPUT_USERS_FINAL, index=False)
	df_edges_final.to_parquet(OUTPUT_EDGES_FINAL, index=False)

	print("Done.")
	print(f"Users df: {OUTPUT_USERS_FINAL}  ({len(df_users_final):,} rows)")
	print(f"Edges df: {OUTPUT_EDGES_FINAL}  ({len(df_edges_final):,} rows)")

	return df_users_final, df_edges_final


if __name__ == "__main__":
	df_users_final, df_edges_final = main()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

edge.csv: 6,335.2 MB -> 2,531.0 MB in 110.9s (57.1 MB/s)


┌──────────────────────┬───────────┬──────────────────────┐
│      source_id       │ relation  │      target_id       │
│       varchar        │  varchar  │       varchar        │
├──────────────────────┼───────────┼──────────────────────┤
│ u980749991491682304  │ followers │ u1480979504696864775 │
│ u105387876           │ following │ u402576793           │
│ u148520716           │ following │ u59653593            │
│ u1276438425457967110 │ following │ u1389155636693381120 │
│ u1445432327367237638 │ following │ u848348952084828160  │
└──────────────────────┴───────────┴──────────────────────┘

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ source_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ relation    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ target_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

Validating relation names...
All 14 configured relation names found in edge file.


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  first hop non-user nodes: 10,017,632
  after expansion:          12,765,885 (11,955,649 tweets, 19,168 lists, 791,068 hashtags)
  total nodes:              12,805,885
  edges:                    22,574,909 in 70.5s
Saving outputs...
Done.
Users df: datasets\final_outputs\df_users_final.parquet  (40,000 rows)
Edges df: datasets\final_outputs\df_edges_final.parquet  (22,574,909 rows)


## Save df_user_final.parquet

In [14]:
df_users_final = pd.read_parquet("./datasets/final_outputs/df_users_final.parquet")
df_users_final.head()

,id,label,split,participation_degree,created_at,description,entities,location,name,pinned_tweet_id,...,public_metrics.tweet_count,public_metrics.listed_count,entities.url.urls,entities.description,entities.url,entities.description.urls,entities.description.mentions,entities.description.hashtags,entities.description.cashtags,withheld.country_codes
0,u2664730894,human,train,1726,2014-07-02 17:56:46+00:00,creative _,NaN,🎈,olawale 💨,NaN,...,1823,0,None,NaN,NaN,None,None,None,None,None
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,,NaN,🇬🇧,Grian,1.143808e+18,...,1400,448,"[{'display_url': 'youtube.com/c/grian', 'end':...",NaN,NaN,None,None,None,None,None
2,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",NaN,None,AK,NaN,...,9194,605,None,NaN,NaN,None,None,None,None,None
3,u1467973039883182090,human,train,1207,2021-12-06 21:44:04+00:00,https://t.co/Hmg5gBvd9A,NaN,None,صارا,NaN,...,146,2,None,NaN,NaN,"[{'display_url': 't.me/BiChatBot?star…', 'end'...",None,None,None,None
4,u234059290,human,train,2267,2011-01-04 19:11:39+00:00,Come for the science (genetics & cell biology)...,NaN,"Salt Lake City,UT, USA",Professor Booty PhD,1.470252e+18,...,91381,116,"[{'display_url': 'profbootyphd.wordpress.com',...",NaN,NaN,None,None,"[{'end': 129, 'start': 112, 'tag': 'BlackLives...",None,None


## Build Tweets File


In [ ]:
try:
	import ijson.backends.yajl2_c as ijson
	print("ijson backend: yajl2_c")
except ImportError:
	import ijson
	print(f"ijson backend: {ijson.backend} (install yajl for a large speedup: conda install yajl)")
import ijson.common

user_ids = set(df_users_final["id"].astype(str))
print(f"Unique sampled user IDs: {len(user_ids):,}")

tweet_files = sorted(glob.glob("./raw_datasets/tweet_*.json"))
output_jsonl = "./datasets/final_outputs/df_tweets_filtered.jsonl"

Path(output_jsonl).parent.mkdir(parents=True, exist_ok=True)

print(f"Tweet files found: {len(tweet_files)}")

chunk = []
writer = None
total_seen = 0
total_matched = 0
truncated = []

with open(output_jsonl, "w", encoding="utf-8", buffering=1024*1024) as out:
	for tweet_file in tweet_files:
		file_seen = 0
		file_matched = 0

		print(f"\nProcessing {tweet_file} ...")

		with open(tweet_file, "rb") as fin:
			try:
				for tweet in ijson.items(fin, "item", use_float=True):
					file_seen += 1
					total_seen += 1

					author_id = tweet.get("author_id")
					if author_id is None:
						continue

					uid = "u" + str(author_id)

					if uid not in user_ids:
						continue

					out.write(json.dumps(tweet, ensure_ascii=False) + "\n")
					file_matched += 1
					total_matched += 1

			except ijson.common.IncompleteJSONError:
				truncated.append(tweet_file)
				print(f"WARNING: {tweet_file} is truncated")

		print(f"Seen: {file_seen:,} | Matched: {file_matched:,}")

print(f"\nDone.")
print(f"Total tweets scanned: {total_seen:,}")
print(f"Total tweets matched: {total_matched:,}")
print(f"Saved to: {output_jsonl}")

if truncated:
	raise RuntimeError(
		f"{len(truncated)} tweet file(s) were truncated and are incomplete. "
		f"Re-download before continuing: {truncated}"
	)

output_parquet = Path("./datasets/final_outputs/df_tweets_final.parquet")

t0 = time.perf_counter()
con.execute(f"""
	COPY (
		SELECT * FROM read_json_auto(
			'{Path(output_jsonl).as_posix()}',
			format='newline_delimited',
			sample_size=-1,
			maximum_object_size=104857600
		)
	)
	TO '{output_parquet.as_posix()}'
	(FORMAT PARQUET, COMPRESSION 'zstd', ROW_GROUP_SIZE 1000000)
""")
print(f"jsonl -> parquet in {time.perf_counter() - t0:,.1f}s")

con.sql(f"DESCRIBE SELECT * FROM read_parquet('{output_parquet.as_posix()}')").show()
con.sql(f"SELECT COUNT(*) FROM read_parquet('{output_parquet.as_posix()}')").show()

ijson backend: yajl2_c
Unique sampled user IDs: 40,000
Tweet files found: 9

Processing ./raw_datasets\tweet_0.json ...
Seen: 10,000,000 | Matched: 598,447

Processing ./raw_datasets\tweet_1.json ...
Seen: 10,000,000 | Matched: 3,262,268

Processing ./raw_datasets\tweet_2.json ...
Seen: 10,000,000 | Matched: 1,244,332

Processing ./raw_datasets\tweet_3.json ...
Seen: 10,000,000 | Matched: 1,255,041

Processing ./raw_datasets\tweet_4.json ...
Seen: 10,000,000 | Matched: 724,027

Processing ./raw_datasets\tweet_5.json ...
Seen: 10,000,000 | Matched: 647,731

Processing ./raw_datasets\tweet_6.json ...
Seen: 10,000,000 | Matched: 532,274

Processing ./raw_datasets\tweet_7.json ...
Seen: 10,000,000 | Matched: 524,151

Processing ./raw_datasets\tweet_8.json ...
Seen: 8,217,457 | Matched: 648,962

Done.
Total tweets scanned: 88,217,457
Total tweets matched: 9,437,233
Saved to: ./datasets/final_outputs/df_tweets_filtered.jsonl


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

jsonl -> parquet in 240.3s
┌─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## Analyse graph connectivity

In [ ]:
df_edges_final = pq.read_table(OUTPUT_EDGES_FINAL).to_pandas(ignore_metadata=True)

codes, uniques = pd.factorize(
	pd.concat([df_edges_final["source_id"], df_edges_final["target_id"]]),
	sort=False,
)
m = len(df_edges_final)
src_code = codes[:m]
tgt_code = codes[m:]

uniques = pd.Index(uniques.astype(str))
n = len(uniques)
first_char = uniques.str[0]

print("=== BASIC COUNTS ===")
print("Total edges:", m)
print("Total nodes:", n)
print("User nodes:", int((first_char == "u").sum()))
print("Tweet nodes:", int((first_char == "t").sum()))
print("List nodes:", int((first_char == "l").sum()))
print("Hashtag nodes:", int((first_char == "h").sum()))
print()

self_loops = int((src_code == tgt_code).sum())
print("Self-loops:", self_loops)
print()

print("=== RELATION COUNTS ===")
print(df_edges_final["relation"].value_counts())
print()

density_directed = m / (n * (n - 1)) if n > 1 else 0.0

print("=== FULL DIRECTED GRAPH DENSITY ===")
print("Directed density:", density_directed)
print()

deg = np.bincount(src_code, minlength=n) + np.bincount(tgt_code, minlength=n)
deg_s = pd.Series(deg, index=uniques, name="total_degree")

print("=== DEGREE STATS (ALL NODES) ===")
print(deg_s.describe())
print()

print("=== DEGREE STATS (USER NODES ONLY) ===")
print(deg_s[first_char == "u"].describe())
print()

is_user = first_char == "u"
uu_mask = is_user[src_code] & is_user[tgt_code]

uu_m = int(uu_mask.sum())
uu_n = len(np.union1d(src_code[uu_mask], tgt_code[uu_mask]))
uu_density = uu_m / (uu_n * (uu_n - 1)) if uu_n > 1 else 0.0

print("=== USER-USER SUBGRAPH ===")
print("User-user edges:", uu_m)
print("User-user nodes:", uu_n)
print("User-user directed density:", uu_density)
print()

adj = coo_matrix(
	(np.ones(m, dtype=np.int8), (src_code, tgt_code)),
	shape=(n, n),
)

n_comp, comp_labels = connected_components(adj, directed=True, connection="weak")
sizes = np.sort(np.bincount(comp_labels))[::-1]

print("=== CONNECTIVITY ===")
print("Number of weakly connected components:", n_comp)
print("Largest weakly connected component size:", int(sizes[0]))
print("Largest WCC ratio:", sizes[0] / n)
print("Top 10 component sizes:", sizes[:10].tolist())
print()

sampled_user_ids = pd.Index(df_users_final["id"].astype(str))
users_with_no_edges = sampled_user_ids.difference(uniques)

print("=== USERS WITH NO EDGES ===")
print("Sampled users with zero edges in final graph:", len(users_with_no_edges))
print("Ratio:", len(users_with_no_edges) / len(sampled_user_ids) if len(sampled_user_ids) else 0)
print()

=== BASIC COUNTS ===
Total edges: 22574909
Total nodes: 12805885
User nodes: 40000
Tweet nodes: 11955649
List nodes: 19168
Hashtag nodes: 791068

Self-loops: 0

=== RELATION COUNTS ===
relation
post          9437233
discuss       7590118
contain       1758261
mentioned     1043855
retweeted     1000974
replied_to     672569
following      313570
like           299090
quoted         184452
membership     130134
followers       88604
followed        34676
pinned          16842
own              4531
Name: count, dtype: int64

=== FULL DIRECTED GRAPH DENSITY ===
Directed density: 1.3765970907058707e-07

=== DEGREE STATS (ALL NODES) ===
count    1.280588e+07
mean     3.525709e+00
std      4.556603e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      6.984400e+04
Name: total_degree, dtype: float64

=== DEGREE STATS (USER NODES ONLY) ===
count    40000.000000
mean       294.267725
std        393.646600
min          1.000000
25%         65.000000